In [3]:
import pandas as pd
import altair as alt
from scipy import stats
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

NAME = "debt_and_bond_yields_2026"   # new name so the original debt_and_bond_yields files aren't overwritten
HIGHLIGHT = ["USA", "GBR", "JPN", "DEU", "AUS", "CHE", "MEX"]
NAMES = {"USA": "United States", "GBR": "UK", "JPN": "Japan", "DEU": "Germany", "AUS": "Australia",
         "CHE": "Switzerland", "MEX": "Mexico", "FRA": "France", "ITA": "Italy", "CAN": "Canada",
         "KOR": "South Korea", "IDN": "Indonesia", "ZAF": "South Africa", "ESP": "Spain", "BEL": "Belgium",
         "PRT": "Portugal", "NLD": "Netherlands", "CZE": "Czechia", "POL": "Poland", "HUN": "Hungary",
         "IRL": "Ireland", "AUT": "Austria"}
SHORT = {"USA": "USA", "GBR": "UK", "JPN": "Japan", "DEU": "Germany", "AUS": "Australia",
         "CHE": "Switzerland", "MEX": "Mexico"}                      # chart labels for the highlighted economies

# ---- 1. Net debt: IMF WEO, 2026 (general government net debt, % of GDP) ----
net_debt = {
    "USA": 98.4,  "GBR": 93.4,  "JPN": 127.0, "DEU": 49.6,  "AUS": 34.0,  "CHE": 17.2,
    "MEX": 53.6,  "FRA": 111.3, "ITA": 128.5, "CAN": 9.3,   "KOR": 7.0,   "IDN": 39.5,
    "ZAF": 76.0,  "ESP": 82.1,  "BEL": 95.5,  "PRT": 80.8,  "NLD": 37.4,  "CZE": 31.3,
    "POL": 53.0,  "HUN": 67.8,  "IRL": 22.5,  "AUT": 65.3,
}
debt = (pd.Series(net_debt, name="net_debt_gdp")
        .rename_axis("iso3").reset_index())

# ---- 2. Yields: LSEG, 10-year benchmark, 3 Nov 2026 ----
yld = pd.Series({
    "USA": 4.91, "GBR": 5.23, "JPN": 2.77, "DEU": 3.32, "AUS": 5.09, "CHE": 0.49,
    "MEX": 9.27, "FRA": None, "ITA": 4.48, "CAN": 3.75, "KOR": 4.00, "IDN": 7.71,
    "ZAF": 8.67, "ESP": 3.93, "BEL": 4.44, "PRT": 3.77, "NLD": None, "CZE": None,
    "POL": 6.30, "HUN": 5.54, "IRL": None, "AUT": None,   # fill the five None values
}, name="yield_10y").rename_axis("iso3").reset_index()

df = yld.merge(debt, on="iso3", how="left")
missing = df[df["yield_10y"].isna() | df["net_debt_gdp"].isna()]
if len(missing):
    print("Dropped for now (missing yield or net debt):", missing["iso3"].tolist())
df = df.dropna(subset=["net_debt_gdp", "yield_10y"]).copy()
df["country"] = df["iso3"].map(NAMES)
n = len(df)

# ---- 3. Regression: kept for the dashed line and for the text under the chart, not drawn on the chart ----
def ols(d):
    r = stats.linregress(d["net_debt_gdp"], d["yield_10y"])
    return r, r.rvalue ** 2

res, r2 = ols(df)
res_xj, r2_xj = ols(df[df["iso3"] != "JPN"])
print(f"All:             n={n}  R2={r2:.2f}  slope={res.slope:.3f}  p={res.pvalue:.2f}")
print(f"Excluding Japan: n={n-1}  R2={r2_xj:.2f}  slope={res_xj.slope:.3f}  p={res_xj.pvalue:.2f}")

fit = pd.DataFrame({"net_debt_gdp": [0.0, 140.0]})
fit["fit"] = res.intercept + res.slope * fit["net_debt_gdp"]

# One frame per layer, only the columns it needs
cols = ["net_debt_gdp", "yield_10y", "country"]
focus_df = df[df["iso3"].isin(HIGHLIGHT)][cols].assign(label=lambda t: df.loc[t.index, "iso3"].map(SHORT)).reset_index(drop=True)
focus_df["group"] = "focus"
others_df = df[~df["iso3"].isin(HIGHLIGHT)][cols].reset_index(drop=True)

# ---- 4. Chart ----
GREY = "#B8BCC8"      # the one explicit colour: the other economies are context, the theme palette has no grey
x = alt.X("net_debt_gdp:Q", title="Net debt to GDP", scale=alt.Scale(domain=[0, 140], nice=False),
          axis=alt.Axis(values=[0, 20, 40, 60, 80, 100, 120, 140], labelExpr="datum.value + '%'", grid=False))
def yenc(field):       # same title, scale and axis in every layer so Altair merges them into one axis
    return alt.Y(f"{field}:Q", title="10-year government bond yield", scale=alt.Scale(domain=[0, 10], nice=False),
                 axis=alt.Axis(values=[0, 2, 4, 6, 8, 10], labelExpr="datum.value + '%'"))
y = yenc("yield_10y")
tip = [alt.Tooltip("country:N", title="Economy"),
       alt.Tooltip("net_debt_gdp:Q", format=".0f", title="Net debt (% of GDP)"),
       alt.Tooltip("yield_10y:Q", format=".2f", title="10-year yield (%)")]

fit_line = alt.Chart(fit).mark_line(strokeDash=[6, 4], strokeWidth=1.8, color="#676A86").encode(
    x=x, y=yenc("fit"))

others = alt.Chart(others_df).mark_circle(size=90, opacity=1, color=GREY).encode(x=x, y=y, tooltip=tip)

focus = alt.Chart(focus_df).mark_circle(size=110, opacity=1).encode(
    x=x, y=y, color=alt.Color("group:N", legend=None), tooltip=tip)                      # theme palette

def label_layer(sub, dy):
    return alt.Chart(sub).mark_text(align="left", baseline="middle", dx=10, dy=dy, fontSize=12, color="#676A86").encode(
        x=x, y=y, text="label:N")
BELOW = ["USA"]                                                   # labels placed under the point
labels = label_layer(focus_df[~focus_df["label"].isin(BELOW)], -9) + label_layer(focus_df[focus_df["label"].isin(BELOW)], 13)

caption = alt.Title(
    text="Source: IMF, World Economic Outlook (general government net debt, 2026); LSEG (10-year benchmark yields, 3 November 2026)",
    subtitle=["Net debt to GDP against 10-year government bond yields, selected economies. Dashed line = linear trend across all economies shown."],
    orient="bottom", anchor="start", fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = alt.layer(fit_line, others, focus, labels).properties(width=400, height=300).configure(
    background="white", font="Circular Std, Helvetica Neue, Helvetica, Arial, sans-serif").configure_view(
    fill="transparent", stroke="transparent")

styles.save(chart, path=".", name=NAME, svg=True)
chart.save(f"{NAME}.png", scale_factor=2)
chart.save(f"{NAME}.html")
chart

Dropped for now (missing yield or net debt): ['FRA', 'NLD', 'CZE', 'IRL', 'AUT']
All:             n=17  R2=0.00  slope=0.002  p=0.88
Excluding Japan: n=16  R2=0.03  slope=0.011  p=0.53


alt.LayerChart(...)

In [ ]:
USA - 4.91
UK - 5.23
Japan - 2.77
Germany - 3.32
Canada - 3.75
Switzerland - 0.49
China - 1.64
Brazil - 15.42
Turkey - 33.06
Italy - 4.48
Australia - 5.09
South Korea - 4.00
Mexico - 9.27
India - 7.32
Indonesia - 7.71
SA - 8.67
Spain - 3.93
Belgium - 4.44
Portugal - 3.77
Greece - 4.27
Poland - 6.30
Hungary - 5.54